# 01 — Preprocessing y Feature Engineering

---
Este notebook ejecuta la sección de Feature Engineering del pipeline.
Carga el dataset imputado, aplica todas las transformaciones de variables y guarda
el DataFrame resultante en `data/processed/df_preprocessed.csv` para uso en los
notebooks siguientes.



## 1. Setup e Importaciones

In [18]:
import warnings; warnings.filterwarnings('ignore')
import numpy  as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

SEED = 42
np.random.seed(SEED)
print('Librerías cargadas')

Librerías cargadas


## 2. Configuración de rutas y parámetros globales

In [19]:
DATA_DIR      = Path('../../')
PROCESSED_DIR = Path('../../data/processed'); PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PRETEC21_GENS  = ['AD14', 'AD15', 'AD16', 'AD17', 'AD18']
TEC21_GENS     = ['AD19', 'AD20']
TARGET         = 'retention'
K_CLUSTERS     = 4
MIN_SILHOUETTE = 0.30
MAX_DB         = 1.50
MIN_AUC        = 0.60
MATCH_HIGH     = 0.15
MATCH_PARTIAL  = 0.35

print(f'DATA_DIR      : {DATA_DIR}')
print(f'PROCESSED_DIR : {PROCESSED_DIR}')

DATA_DIR      : ../..
PROCESSED_DIR : ../../data/processed


## 3. Carga del Dataset

In [21]:
csv_path = DATA_DIR / 'dataset_imputed.csv'
df_raw   = pd.read_csv(csv_path, low_memory=False)

print(f'\nDistribución por generación:')
print(df_raw['generation'].value_counts().sort_index().to_string())

counts = df_raw[TARGET].value_counts()
print(f'\nDistribución target:')
print(f'  Retuvo  (1): {counts.get(1,0):>6,}  ({counts.get(1,0)/len(df_raw)*100:.1f}%)')
print(f'  Desertó (0): {counts.get(0,0):>6,}  ({counts.get(0,0)/len(df_raw)*100:.1f}%)')


Distribución por generación:
generation
AD14    10143
AD15    10041
AD16    10742
AD17    10788
AD18    11296
AD19    12199
AD20    12308

Distribución target:
  Retuvo  (1): 70,704  (91.2%)
  Desertó (0):  6,813  (8.8%)


## 4. Feature Engineering

In [22]:
# Normalización de admission_test 
def norm_admission_test(val):
    if pd.isna(val): return np.nan
    try:
        f = float(val)
        return max(0.0, (f - 400) / 1200.0) if f > 100 else f / 100.0
    except: return np.nan

EDU_ORD = {
    'No information': -1, 'MISSING': -1,
    'No degree': 0, 'Undergraduate degree': 1, 'Master degree': 2, 'PhD': 3,
}

df = df_raw.copy()
if 'level' in df.columns:
    df = df[df['level'] == 'Undergraduate'].copy()
    print(f'Filtrado nivel universitario: {len(df):,} registros')

DROP_COLS = [
    'student.id', 'level', 'average.first.period', 'failed.subject.first.period',
    'dropped.subject.first.period', 'dropout.semester', 'program', 'id.school.origin',
    'scholarship.type', 'school.cost', 'parents.exatec', 'father.exatec', 'mother.exatec',
    'father.education.complete', 'father.education.summary',
    'mother.education.complete', 'mother.education.summary',
    'scholarship.perc', 'loan.perc', 'educational.model',
]
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

# admission_test_norm 
if 'admission.test' in df.columns:
    df['admission_test_norm'] = df['admission.test'].apply(norm_admission_test)
    df.drop(columns=['admission.test'], inplace=True)
if 'admission_test_norm' not in df.columns:
    df['admission_test_norm'] = np.nan
df['admission_test_norm'].fillna(df['admission_test_norm'].median(), inplace=True)

#  first_gen_enc 
def enc_first_gen(v):
    s = str(v).strip() if pd.notna(v) else ''
    return 1 if s == 'Yes' else (0 if s == 'No' else -1)
if 'first.generation' in df.columns:
    df['first_gen_enc'] = df['first.generation'].apply(enc_first_gen)
    df.drop(columns=['first.generation'], inplace=True)
elif 'first_gen_enc' not in df.columns:
    df['first_gen_enc'] = -1

# educ_padres_max 
if 'max.degree.parents' in df.columns:
    df['educ_padres_max'] = df['max.degree.parents'].map(EDU_ORD).fillna(-1).astype(int)
    df.drop(columns=['max.degree.parents'], inplace=True)
elif 'educ_padres_max' not in df.columns:
    df['educ_padres_max'] = -1

# apoyo_financiero 
if 'total.scholarship.loan' in df.columns:
    df.rename(columns={'total.scholarship.loan': 'apoyo_financiero'}, inplace=True)
elif 'apoyo_financiero' not in df.columns:
    df['apoyo_financiero'] = 0.0

# has_extracurriculars 
if 'has_extracurriculars' not in df.columns:
    df['has_extracurriculars'] = df['has_life_activities'].copy() if 'has_life_activities' in df.columns else 0

ACTIVITY_COLS = ['physical.education','cultural.diffusion','student.society',
                 'total.life.activities','athletic.sports','art.culture',
                 'student.society.leadership','life.work.mentoring','wellness.activities']
df.drop(columns=[c for c in ACTIVITY_COLS + ['has_life_activities'] if c in df.columns], inplace=True)

# género 
if 'gender' in df.columns:
    df['is_male'] = df['gender'].map({'Male':1,'Female':0}).fillna(0).astype(int)
    df.drop(columns=['gender'], inplace=True)
elif 'is_male' not in df.columns:
    df['is_male'] = 0

# prepa tec 
if 'tec.no.tec' in df.columns:
    df['estuvo_prepa_tec'] = df['tec.no.tec'].map({'TEC':1,'NO TEC':0}).fillna(0).astype(int)
    df.drop(columns=['tec.no.tec'], inplace=True)
elif 'estuvo_prepa_tec' not in df.columns:
    df['estuvo_prepa_tec'] = 0

# nivel socioeconómico 
if 'socioeconomic.level' in df.columns:
    df['socioec_enc'] = df['socioeconomic.level'].map(
        {**{'No information':0,'MISSING':0},**{f'Level {i}':i for i in range(1,8)}}
    ).fillna(0).astype(int)
    df.drop(columns=['socioeconomic.level'], inplace=True)
elif 'socioec_enc' not in df.columns:
    df['socioec_enc'] = 0

# rezago social 
if 'social.lag' in df.columns:
    df['social_lag_enc'] = df['social.lag'].map(
        {'No information':0,'MISSING':0,'Low':1,'Medium':2,'High':3}
    ).fillna(0).astype(int)
    df.drop(columns=['social.lag'], inplace=True)
elif 'social_lag_enc' not in df.columns:
    df['social_lag_enc'] = 0

# campus
if 'school' in df.columns:
    le_school = LabelEncoder()
    df['school_enc'] = le_school.fit_transform(df['school'].fillna('Unknown').astype(str))
    df.drop(columns=['school'], inplace=True)
elif 'school_enc' not in df.columns:
    df['school_enc'] = 0

# región 
if 'region' in df.columns:
    le_region = LabelEncoder()
    df['region_enc'] = le_region.fit_transform(df['region'].fillna('Unknown').astype(str))
    df.drop(columns=['region'], inplace=True)
elif 'region_enc' not in df.columns:
    df['region_enc'] = 0

# extranjero 
if 'foreign' in df.columns:
    foreign_dummies = pd.get_dummies(df['foreign'], prefix='foreign', drop_first=True, dtype=int)
    df = pd.concat([df, foreign_dummies], axis=1)
    df.drop(columns=['foreign'], inplace=True)

df.drop(columns=[c for c in ['zone.type'] if c in df.columns], inplace=True)

print(f'\nFeature engineering completado. Shape: {df.shape}')

Filtrado nivel universitario: 77,517 registros

Feature engineering completado. Shape: (77517, 29)


## 5. Guardar DataFrame procesado

In [23]:
out = PROCESSED_DIR / 'df_preprocessed.csv'
df.to_csv(out, index=False)

print(f'  Guardado: {out}')
print(f'  Shape   : {df.shape}')
print(f'  Columnas: {list(df.columns)}')
print('\nSiguiente a correr: clustering/02_kmeans_independiente.ipynb')

  Guardado: ../../data/processed/df_preprocessed.csv
  Shape   : (77517, 29)
  Columnas: ['generation', 'age', 'PNA', 'online.test', 'english.evaluation', 'admission.rubric', 'general.math.eval', 'retention', 'FTE', 'apoyo_financiero', 'regime', 'took_admission_test', 'first_gen_present', 'parents_edu_present', 'has_socioeconomic_data', 'has_social_lag_data', 'has_zone_data', 'admission_test_norm', 'first_gen_enc', 'educ_padres_max', 'has_extracurriculars', 'is_male', 'estuvo_prepa_tec', 'socioec_enc', 'social_lag_enc', 'school_enc', 'region_enc', 'foreign_Yes: Foreigner', 'foreign_Yes: National']

Siguiente a correr: clustering/02_kmeans_independiente.ipynb
